#  Qdrant initiation

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")

# Load point paths for collection

In [2]:
from glob import glob
from tqdm import tqdm
import json

COLLECTION_NAME = "test"
point_paths = glob(rf"..\collections\{COLLECTION_NAME}\*.json") # Points in JSON format, compatible with Qdrant upsertion

# Collection configuration

In [3]:
dense_siglip2_params = models.VectorParams(
    size=1152,
    distance=models.Distance.COSINE,
    on_disk=True
)

dense_bgem3_params = models.VectorParams(
    size=1024,
    distance=models.Distance.COSINE,
    on_disk=True
)

sparse_params = models.SparseVectorParams(
    index=models.SparseIndexParams(on_disk=True),
    modifier=models.Modifier.IDF
)

hnsw_params = models.HnswConfigDiff(
    m=32,
    ef_construct=200,
    full_scan_threshold=10000
)

quantization_params = models.ScalarQuantization(
    scalar=models.ScalarQuantizationConfig(
        type=models.ScalarType.INT8,
        quantile=1.0,
        always_ram=True
    )
)

In [5]:
if client.collection_exists(collection_name=COLLECTION_NAME):
    confirm_deletion = input('The collection already existed, do you want to replace it (y/n)? ')
    if confirm_deletion.lower() == 'y':
        client.delete_collection(collection_name=COLLECTION_NAME)
        print('Deletion complete. Run the command again')
    elif confirm_deletion.lower() == 'n':
        print('Deletion cancelled')
    else:
        print('Invalid input')
else:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            'dense_siglip2': dense_siglip2_params,
            'dense_bgem3': dense_bgem3_params,
        },
        sparse_vectors_config={'sparse_bgem3': sparse_params},
        hnsw_config=hnsw_params
    )

# Upsertion

In [6]:
# Testing real upload speed for future process
point_paths = point_paths * 50

In [6]:
BATCH_SIZE = 1024
NUM_PARALLEL = 32
for i in tqdm(range(0, len(point_paths), BATCH_SIZE)):
    points = []
    j = i + BATCH_SIZE if i + BATCH_SIZE < len(point_paths) else len(point_paths)
    for k, path in enumerate(point_paths[i:j]):
        with open(path, 'r', encoding='utf-8') as f:
            point = json.load(f)

        # Reformat to Qdrant point
        point = models.PointStruct(
            id=i+k+1,
            vector={
                'dense_siglip2': point['vector']['dense_siglip2'],
                'dense_bgem3': point['vector']['dense_bgem3'],
                'sparse_bgem3': models.SparseVector(
                    indices=point['vector']['sparse_bgem3']['token_ids'],
                    values=point['vector']['sparse_bgem3']['weights']
                ),
                # 'multi': point['vector']["multi"]
            },
            payload=point['payload']
        )
        points.append(point)

    client.upload_points(
        collection_name=COLLECTION_NAME,
        points=points,
        batch_size=BATCH_SIZE // (NUM_PARALLEL * 2),
        parallel=NUM_PARALLEL
    )

100%|██████████| 8/8 [01:56<00:00, 14.51s/it]
